<a href="https://colab.research.google.com/github/rtajeong/M3_2026/blob/main/lab53_sentiment_analysis_naver_movie_rev3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

네이버영화평점
==
- 감성분석
- 네이버 영화평점 (Naver sentiment movie corpus v.1.0) 데이터(https://github.com/e9t/nsmc)
- 영화 리뷰 20만건이 저장됨. 각 평가 데이터는 0(부정), 1(긍정)으로 label 됨.

### 한글 자연어 처리
- KoNLPy(“코엔엘파이”라고 읽습니다)는 한국어 정보처리를 위한 파이썬 패키지입니다.
- konlpy 패키지에서 제공하는 Twitter라는 문서 분석 라이브러리 사용 (트위터 분석 뿐 아니라 한글 텍스트
  처리도 가능)
- colab 사용 권장

# 로지스틱회귀를 이용한 감성분석

In [30]:
!pip install konlpy

In [31]:
# 네이버 영화 평점 데이터 다운로드
#!curl -L https://bit.ly/2X9Owwr -o ratings_train.txt
#!curl -L https://bit.ly/2WuLd5I -o ratings_test.txt

- 다운로드 사이트 링크가 없어져 저장해 둔 파일을 직접 업로드한다.

In [32]:
import konlpy
import pandas as pd
from konlpy.tag import Twitter, Okt
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
# from sklearn.pipeline import make_pipeline
# import pickle
# import os.path

# 데이터 로드

# keep_default_na=True (기본값) → Pandas가 기본적으로 정의한 NA 문자열 집합을 그대로 사용
#    → 즉 "NA", "NaN", "null" 같은 값은 자동으로 NaN으로 변환.
# keep_default_na=False → Pandas의 기본 NA 문자열 집합을 사용하지 않음.
#    → "NA", "NaN" 같은 값도 그냥 문자열로 남는다.

df_train = pd.read_csv('ratings_train.txt', delimiter='\t', keep_default_na=False)
df_test = pd.read_csv('ratings_test.txt', delimiter='\t', keep_default_na=False)

In [33]:
df_train.head(2)

,id,document,label
0,9976970,아 더빙.. 진짜 짜증나네요 목소리,0
1,3819312,흠...포스터보고 초딩영화줄....오버연기조차 가볍지 않구나,1


In [34]:
df_test.head(2)

,id,document,label
0,6270596,굳 ㅋ,1
1,9274899,GDNTOPCLASSINTHECLUB,0


In [35]:
text_train, y_train = df_train['document'].values, df_train['label'].values
text_test, y_test = df_test['document'].values, df_test['label'].values

In [36]:
text_train.shape, text_test.shape   # too big

((150000,), (50000,))

In [37]:
text_train, y_train = text_train[:2000], y_train[:2000]
text_test, y_test = text_test[:1000], y_test[:1000]

In [38]:
text_train.shape, text_test.shape

((2000,), (1000,))

In [39]:
from konlpy.tag import Okt

def okt_tokenizer(text):
    return Okt().morphs(text)


In [40]:
cv = TfidfVectorizer(tokenizer=okt_tokenizer, max_features = 1000, min_df=5)
x_train = cv.fit_transform(text_train)
x_test = cv.transform(text_test)
print(x_train.shape, x_test.shape, y_train.shape, y_test.shape)

/usr/local/lib/python3.12/dist-packages/sklearn/feature_extraction/text.py:517: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


(2000, 794) (1000, 794) (2000,) (1000,)


In [41]:
print(cv.vocabulary_)
print(cv.get_feature_names_out()[-10:])

{'아': np.int64(432), '더빙': np.int64(211), '..': np.int64(13), '진짜': np.int64(693), '흠': np.int64(791), '...': np.int64(14), '포스터': np.int64(742), '보고': np.int64(326), '영화': np.int64(520), '줄': np.int64(680), '....': np.int64(15), '연기': np.int64(512), '너': np.int64(174), '추천': np.int64(716), '한': np.int64(760), '다': np.int64(194), '이야기': np.int64(579), '솔직히': np.int64(402), '재미': np.int64(622), '는': np.int64(190), '없다': np.int64(484), '평점': np.int64(741), '그': np.int64(119), '의': np.int64(555), '가': np.int64(60), '!': np.int64(0), '에서': np.int64(496), '했던': np.int64(777), '너무나도': np.int64(177), '막': np.int64(261), '3': np.int64(24), '세': np.int64(392), '부터': np.int64(360), '초등학교': np.int64(708), '1': np.int64(20), '8': np.int64(28), '.': np.int64(12), 'ㅋㅋㅋ': np.int64(50), '별': np.int64(321), '반개': np.int64(314), '도': np.int64(213), '아까': np.int64(433), '움': np.int64(542), '원작': np.int64(546), '긴장감': np.int64(145), '을': np.int64(553), '제대로': np.int64(656), '했다': np.int64(776), '아깝다': np.

In [42]:
y_train = y_train.astype(int)
y_test = y_test.astype(int)
lr = LogisticRegression()
lr.fit(x_train, y_train)
print("train score: ", lr.score(x_train, y_train))
print("test score: ", lr.score(x_test, y_test))

train score:  0.855
test score:  0.745


# 불용어 처리
- 한국어  불용어 확인은 형태소 분석 라이브러리인 KonLPy 를 이용하면 됨.
- (예) 한국어 품사 중 조사를 추출하는 예
- pos (part-of-speech): 품사 (명사, 동사, ...)

- norm: 오타수정, stem: 어근 찾기

In [43]:
def okt_tokenizer2(text):
    word_tags = Okt().pos(text, norm=True, stem=True)
    words = [word[0] for word in word_tags if word[1]!="Josa" and word[1]!="Punctuation"]
    return words

cv = TfidfVectorizer(tokenizer=okt_tokenizer2, max_features = 500, min_df=5)
x_train = cv.fit_transform(text_train)
x_test = cv.transform(text_test)

lr = LogisticRegression()
lr.fit(x_train,y_train)
print("훈련 데이터 점수 : ", lr.score(x_train, y_train))
print("테스트 데이터 점수 : ", lr.score(x_test, y_test))

/usr/local/lib/python3.12/dist-packages/sklearn/feature_extraction/text.py:517: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


훈련 데이터 점수 :  0.835
테스트 데이터 점수 :  0.76


------------------